In [29]:
import re
import fitz


def extract_text_from_pdf(uploaded_file):

    if isinstance(uploaded_file, str):
        doc = fitz.open(uploaded_file)

    elif hasattr(uploaded_file, "read"):
        pdf_bytes = uploaded_file.read()
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    else:
        doc = fitz.open(stream=uploaded_file, filetype="pdf")

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    return text


def clean_text(text):
    """Clean extracted text"""
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s.,@+\-/#()]", " ", text)
    return text.strip()


def extract_skills_and_experience(cv_text):
    """استخراج الأجزاء المهمة فقط من الـ CV لمساعدة الموديل"""
    clean = cv_text.lower()

    keywords = ["skills", "experience", "projects", "technical skills"]
    important_parts = []

    lines = clean.split("\n")
    for line in lines:
        if any(kw in line for kw in keywords) or len(line.strip()) < 50:
            important_parts.append(line)

    extracted = " ".join(important_parts)
    return extracted if len(extracted) > 100 else cv_text


def process_cv(uploaded_file):
    """Main function for Task 1"""
    # 1. استخراج النص الخام من الـ PDF
    raw_text = extract_text_from_pdf(uploaded_file)

    # 2. تنظيف النص العام
    cleaned_text = clean_text(raw_text)

    # 3. التركيز على الأقسام المهمة (Skills & Experience)
    focused_text = extract_skills_and_experience(raw_text)
    cleaned_focused_text = clean_text(focused_text)

    # 4. إرجاع النتائج للـ Backend لاستخدامها في حساب الـ Score والـ LLM
    return {
        "raw_text": raw_text,
        "cleaned_text": cleaned_text,
        "focused_text": cleaned_focused_text,
    }

In [21]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# 1. تحميل الموديل الأصلي والداتا سيت
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
ds = load_dataset("AzharAli05/Resume-Screening-Dataset", split="train")

# 2. تجهيز داتا التدريب (تحويل select لـ 1.0 و reject لـ 0.0)
train_examples = []
for row in ds:
    # إعطاء قيم مستهدفة (1 للقبول و 0 للرفض)
    label = 1.0 if row["Decision"] == "select" else 0.0
    train_examples.append(
        InputExample(
            texts=[row["Resume"], row["Job_Description"]], label=label
        )
    )

# 3. إعداد الـ DataLoader والـ Loss Function
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

# 4. بدء التدريب (2 Epochs بس هتاخد أسبوع/دقيقتين على حسب الـ CPU/GPU)
print("⏳ جاري تدريب الموديل على البيانات (Fine-Tuning)...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=10,
)
print("✅ تم التدريب بنجاح!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3175.75it/s]


⏳ جاري تدريب الموديل على البيانات (Fine-Tuning)...


f:\NLP_Final_project\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.235755
1000,0.218798


✅ تم التدريب بنجاح!


In [43]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. تحميل الموديل مرة واحدة
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def generate_embedding(text):
    """تحويل النص لـ Vector Embedding"""
    return embedding_model.encode(text)


def calculate_cosine_similarity(vec1, vec2):
    """حساب الـ Cosine Similarity"""
    sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return float(sim)


def get_match_score(cv_cleaned_text, job_description_text):
    """حساب نسبة التوافق المباشرة من الـ Raw Cosine Similarity"""
    cleaned_jd = clean_text(job_description_text)

    # 1. توليد الـ Embeddings
    cv_embedding = generate_embedding(cv_cleaned_text)
    jd_embedding = generate_embedding(cleaned_jd)

    # 2. حساب نسبة التشابه الخام
    raw_similarity = calculate_cosine_similarity(cv_embedding, jd_embedding)

    # 3. تحويل القيمة الخام (من 0 إلى 1) لنسبة مئوية مباشرة (0% إلى 100%)
    final_score = max(0.0, min(100.0, raw_similarity * 100))

    return round(final_score, 2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4544.26it/s]


In [46]:
# ==========================================
# Test Task 1 + Task 2
# ==========================================

pdf_path = r"F:\NLP_Final_project\test.pdf"

job_description = """
Machine Learning Engineer

Location:
Cairo, Egypt

Requirements:

Bachelor's degree in Computer Science or Artificial Intelligence.
2+ years of experience in Machine Learning.
Strong Python programming skills.
Experience with TensorFlow.
Experience with Scikit-learn.
Strong SQL knowledge.
Experience with Docker.
Experience deploying ML applications on AWS.
Experience building REST APIs using Flask.
Experience using Git and GitHub.
Strong communication skills.
Teamwork.
Problem-solving skills.

Responsibilities:

Develop and train machine learning models.
Build AI applications using Python.
Deploy ML models using Docker and AWS.
Build REST APIs using Flask.
Work with SQL databases.
Collaborate with software engineers.
Improve model accuracy and performance.
"""

# تشغيل التاسك الأول والثاني
result = run_first_and_second_party_tasks(
    pdf_path,
    job_description
)

print("=" * 70)
print("Resume Analyzer Result")
print("=" * 70)

print(f"Match Score : {result['match_score']}%")

print("\n")

print("=" * 70)
print("Extracted Resume Text")
print("=" * 70)

print(result["cleaned_text"][:1000])

print("\n")

print("=" * 70)
print(f"Characters : {len(result['cleaned_text'])}")
print(f"Words      : {len(result['cleaned_text'].split())}")
print("=" * 70)

Resume Analyzer Result
Match Score : 75.91%


Extracted Resume Text
Name  Ahmed Mohamed Email  ahmed.ml@gmail.com Phone  +20 1012345678 Location  Cairo, Egypt -------------------------------------------------- PROFESSIONAL SUMMARY Machine Learning Engineer with 3 years of experience developing, training, and deploying machine learning models. Strong background in Python, TensorFlow, Scikit-learn, SQL, Docker, AWS, Git, and Flask. Experienced in building end-to-end AI applications and collaborating with cross-functional teams. -------------------------------------------------- EDUCATION Bachelor of Computer Science Faculty of Computers and Artificial Intelligence 2020 - 2024 -------------------------------------------------- TECHNICAL SKILLS Programming  - Python - SQL Machine Learning  - TensorFlow - Scikit-learn - Pandas - NumPy Deployment  - Docker - AWS - Flask Version Control  - Git - GitHub -------------------------------------------------- WORK EXPERIENCE Machine Learning Enginee